# RBF-SVM vs neural participant disagreement
Examines the next question raised by the participant-difficulty analysis:

> **When the 3MT–TMC neural model is wrong, is the RBF-SVM right?**

That tells us whether the SVM and the evidential neural model make **complementary errors**. If the SVM rescues many neural errors, a later SVM–TMC / SVM–Hybrid fusion experiment is empirically motivated. If both models fail on the same participants, fusion is less likely to solve the main problem and the difficult cases themselves deserve closer clinical / label analysis.

## Analysis

- auto-discovers the saved RBF-SVM out-of-fold prediction files if they exist on JHub;
- supports either the individual fold CSVs or the pooled/all-baseline CSV written by the classical-ML notebook;
- compares the SVM against **small, medium, and full** neural architectures;
- uses a **strict matched-seed comparison** on the common seeds **17, 42, 73**;
- separately reports the five-seed neural ensemble as a descriptive secondary result;
- counts:
  - both correct,
  - **SVM rescues neural**,
  - **neural rescues SVM**,
  - both wrong;
- breaks disagreement down by **fold** and **target class**;
- identifies participants all three neural architectures miss and asks how many the SVM rescues;
- joins uncertainty, 3MT–TMC disagreement, modality availability, and prepared participant covariates where available;
- saves participant-level tables and figures.

## Important methodological note

The RBF-SVM baseline currently uses three final seeds (**17, 42, 73**), whereas the neural experiments use five (**17, 42, 73, 101, 137**). Therefore the **main model-to-model disagreement analysis uses the three common seeds**. The five-seed neural ensemble is shown only as a secondary descriptive comparison.

This is a **post-hoc outer-test diagnostic analysis**. Do not tune a fusion weight from these test results. If the error patterns look complementary, the next fusion experiment should select its weight using training/validation predictions only.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
)

pd.set_option("display.max_columns", 250)
pd.set_option("display.width", 200)

# JHub paths

PROJECT_ROOT = Path("/home/jovyan/thesis_data/adni_mri/adni_mri")
RUNS_ROOT = Path("/home/jovyan/thesis_runs")

INPUT_ROOT = (
    PROJECT_ROOT
    / "models"
    / "final_task_ready_inputs"
    / "mci_prognosis"
)

ARCHITECTURE_ROOTS = {
    "small_1.13M": (
        RUNS_ROOT
        / "gated_cmt_fixed_equal_fusion_small_v1_md050"
    ),
    "medium_2.51M": (
        RUNS_ROOT
        / "gated_cmt_fixed_equal_fusion_medium_v1_md050"
    ),
    "full_4.43M": (
        RUNS_ROOT
        / "gated_cmt_fixed_equal_fusion_full_arch_md050"
    ),
}

COMMON_SEEDS = [17, 42, 73]
ALL_NEURAL_SEEDS = [17, 42, 73, 101, 137]
FOLDS = [0, 1, 2, 3, 4]
THRESHOLD = 0.50

ANALYSIS_ROOT = RUNS_ROOT / "svm_vs_neural_participant_disagreement_v1"
TABLES_DIR = ANALYSIS_ROOT / "tables"
FIGURES_DIR = ANALYSIS_ROOT / "figures"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Manual override
# Leave as None for auto-discovery.
#
# Optional explicit path override.
#   1) the folder containing svm_rbf_fold_0_seed_17_test_predictions.csv
#      OR
#   2) all_baseline_out_of_fold_predictions.csv
#      OR
#   3) svm_rbf_seed_17_pooled_out_of_fold_predictions.csv
#
SVM_SOURCE_OVERRIDE = Path(
    "/home/jovyan/thesis_runs/svm_rbf_predictions"
)

print("Project root:", PROJECT_ROOT)
print("Runs root:", RUNS_ROOT)
print("Common seeds:", COMMON_SEEDS)
print("Neural five-seed set:", ALL_NEURAL_SEEDS)
print("Analysis output:", ANALYSIS_ROOT)


## 1. Helpers and robust SVM file discovery

The original baseline notebook saved SVM predictions with the columns:

- `MODEL`
- `SEED`
- `FOLD`
- `RID`
- `TARGET`
- `PROB_PMCI`
- `PREDICTED_CLASS_0P5`

The analysis accepts that format whether predictions are stored as individual fold files, pooled per-seed files, or inside `all_baseline_out_of_fold_predictions.csv`.


In [ ]:
def require_columns(frame, columns, context="dataframe"):
    missing = [c for c in columns if c not in frame.columns]
    if missing:
        raise KeyError(
            f"{context}: missing required columns {missing}. "
            f"Available columns: {frame.columns.tolist()}"
        )

def safe_auc(y_true, y_prob):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    if np.unique(y_true).size < 2:
        return np.nan
    return float(roc_auc_score(y_true, y_prob))

def safe_ap(y_true, y_prob):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    if np.unique(y_true).size < 2:
        return np.nan
    return float(average_precision_score(y_true, y_prob))

def candidate_search_roots():
    roots = [
        RUNS_ROOT,
        PROJECT_ROOT / "models",
        Path("/home/jovyan/thesis_code"),
    ]
    return [p for p in roots if p.exists()]

def discover_svm_source():
    """Find the baseline SVM predictions without scanning MRI arrays."""

    if SVM_SOURCE_OVERRIDE is not None:
        override = Path(SVM_SOURCE_OVERRIDE)
        if not override.exists():
            raise FileNotFoundError(
                f"SVM_SOURCE_OVERRIDE does not exist: {override}"
            )
        return override

    roots = candidate_search_roots()

    # First preference: one combined OOF file.
    combined_name = "all_baseline_out_of_fold_predictions.csv"

    for root in roots:
        matches = list(root.rglob(combined_name))
        if matches:
            print("Found combined baseline prediction file(s):")
            for p in matches:
                print(" -", p)
            return matches[0]

    # Second preference: pooled SVM seed file.
    pooled_name = "svm_rbf_seed_17_pooled_out_of_fold_predictions.csv"

    for root in roots:
        matches = list(root.rglob(pooled_name))
        if matches:
            print("Found pooled SVM prediction file(s):")
            for p in matches:
                print(" -", p)
            return matches[0].parent

    # Third preference: individual fold files.
    fold_name = "svm_rbf_fold_0_seed_17_test_predictions.csv"

    for root in roots:
        matches = list(root.rglob(fold_name))
        if matches:
            print("Found individual SVM fold prediction file(s):")
            for p in matches:
                print(" -", p)
            return matches[0].parent

    return None

def normalize_svm_frame(frame, context):
    require_columns(
        frame,
        ["SEED", "FOLD", "RID", "TARGET", "PROB_PMCI"],
        context,
    )

    out = frame.copy()

    if "MODEL" in out.columns:
        out = out.loc[
            out["MODEL"].astype(str).str.lower() == "svm_rbf"
        ].copy()

    out["SEED"] = out["SEED"].astype(int)
    out["FOLD"] = out["FOLD"].astype(int)
    out["RID"] = out["RID"].astype(int)
    out["TARGET"] = out["TARGET"].astype(int)
    out["PROB_PMCI"] = out["PROB_PMCI"].astype(float)

    return out

def load_svm_predictions(source):
    """Load SVM OOF predictions from combined file, pooled files, or fold files."""

    source = Path(source)

    # Case A: a CSV file was provided.
    if source.is_file():
        frame = pd.read_csv(source)
        frame = normalize_svm_frame(frame, str(source))

        # If this is a pooled single-seed file, load the sibling pooled seed files.
        if source.name.startswith("svm_rbf_seed_"):
            frames = []
            for seed in COMMON_SEEDS:
                path = source.parent / (
                    f"svm_rbf_seed_{seed}_pooled_out_of_fold_predictions.csv"
                )
                if not path.exists():
                    raise FileNotFoundError(path)
                f = normalize_svm_frame(pd.read_csv(path), str(path))
                frames.append(f)
            frame = pd.concat(frames, ignore_index=True)

    # Case B: a directory was provided / discovered.
    elif source.is_dir():
        # Prefer pooled seed files if present.
        pooled_probe = (
            source
            / "svm_rbf_seed_17_pooled_out_of_fold_predictions.csv"
        )

        if pooled_probe.exists():
            frames = []
            for seed in COMMON_SEEDS:
                path = (
                    source
                    / f"svm_rbf_seed_{seed}_pooled_out_of_fold_predictions.csv"
                )
                if not path.exists():
                    raise FileNotFoundError(path)
                f = normalize_svm_frame(pd.read_csv(path), str(path))
                frames.append(f)

            frame = pd.concat(frames, ignore_index=True)

        else:
            frames = []

            for seed in COMMON_SEEDS:
                for fold in FOLDS:
                    path = (
                        source
                        / f"svm_rbf_fold_{fold}_seed_{seed}_test_predictions.csv"
                    )
                    if not path.exists():
                        raise FileNotFoundError(path)

                    f = normalize_svm_frame(pd.read_csv(path), str(path))
                    frames.append(f)

            frame = pd.concat(frames, ignore_index=True)

    else:
        raise FileNotFoundError(source)

    frame = frame.loc[
        frame["SEED"].isin(COMMON_SEEDS)
    ].copy()

    duplicate = frame.duplicated(["SEED", "RID"], keep=False)
    if duplicate.any():
        bad = frame.loc[
            duplicate,
            ["SEED", "FOLD", "RID"],
        ].sort_values(["SEED", "RID"])

        raise ValueError(
            "Duplicate participant within one SVM seed's OOF predictions. "
            f"Example:\n{bad.head(20)}"
        )

    expected_rows = 544 * len(COMMON_SEEDS)

    print("Loaded SVM rows:", len(frame))
    print("Expected for 3 seeds:", expected_rows)
    print("Unique participants:", frame["RID"].nunique())
    print("Seeds:", sorted(frame["SEED"].unique().tolist()))
    print("Folds:", sorted(frame["FOLD"].unique().tolist()))

    return frame

def load_neural_predictions(
    experiment_root,
    architecture_name,
    seeds,
):
    frames = []

    for seed in seeds:
        for fold in FOLDS:
            path = (
                experiment_root
                / f"seed_{seed}"
                / "mci_prognosis"
                / f"fold_{fold}"
                / "predictions"
                / "test_predictions.csv"
            )

            if not path.exists():
                continue

            frame = pd.read_csv(path)

            require_columns(
                frame,
                ["RID", "TARGET", "FINAL_P_pMCI"],
                str(path),
            )

            frame = frame.copy()
            frame["SEED"] = int(seed)
            frame["FOLD"] = int(fold)
            frame["ARCHITECTURE"] = architecture_name

            frames.append(frame)

    if not frames:
        raise FileNotFoundError(
            f"No neural prediction files found under {experiment_root}"
        )

    out = pd.concat(frames, ignore_index=True)

    if out.duplicated(["SEED", "RID"]).any():
        raise ValueError(
            f"{architecture_name}: duplicate RID within a seed's OOF predictions."
        )

    return out

def aggregate_seed_ensemble(
    frame,
    probability_column,
    seeds,
    uncertainty_column=None,
):
    use = frame.loc[frame["SEED"].isin(seeds)].copy()

    require_columns(
        use,
        ["RID", "TARGET", "FOLD", "SEED", probability_column],
        "ensemble input",
    )

    agg = {
        "TARGET": "first",
        "FOLD": "first",
        "SEED": "nunique",
        probability_column: ["mean", "std"],
    }

    if uncertainty_column is not None and uncertainty_column in use.columns:
        agg[uncertainty_column] = ["mean", "std"]

    optional_cols = [
        "TMC_UNCERTAINTY",
        "THREE_MT_UNCERTAINTY",
        "TMC_P_pMCI",
        "THREE_MT_P_pMCI",
        "ORIGINAL_MODALITY_COUNT",
        "EFFECTIVE_MODALITY_COUNT",
    ]

    for col in optional_cols:
        if col in use.columns and col not in agg:
            agg[col] = ["mean", "std"]

    result = use.groupby("RID").agg(agg)
    result.columns = [
        "__".join([str(x) for x in tup if str(x)])
        for tup in result.columns
    ]
    result = result.reset_index()

    rename = {
        "TARGET__first": "TARGET",
        "FOLD__first": "FOLD",
        "SEED__nunique": "N_SEEDS",
        f"{probability_column}__mean": "P_MEAN",
        f"{probability_column}__std": "P_SD",
    }

    if uncertainty_column is not None:
        rename[f"{uncertainty_column}__mean"] = "UNCERTAINTY_MEAN"
        rename[f"{uncertainty_column}__std"] = "UNCERTAINTY_SD"

    result = result.rename(columns=rename)

    result["PRED_CLASS"] = (
        result["P_MEAN"] >= THRESHOLD
    ).astype(int)

    result["CORRECT"] = (
        result["PRED_CLASS"] == result["TARGET"]
    ).astype(int)

    result["CORRECT_CLASS_PROBABILITY"] = np.where(
        result["TARGET"] == 1,
        result["P_MEAN"],
        1.0 - result["P_MEAN"],
    )

    return result


## 2. Locate and load the saved RBF-SVM predictions

 First check whether their output folder was copied to JHub. Set `SVM_SOURCE_OVERRIDE` in the configuration cell to the exact file/folder and rerun from there.


In [ ]:
svm_source = discover_svm_source()

if svm_source is None:
    print("SVM prediction files were NOT found automatically.")
    print()
    print("Searched under:")
    for root in candidate_search_roots():
        print(" -", root)
    print()
    print("Expected one of these:")
    print(" - all_baseline_out_of_fold_predictions.csv")
    print(" - svm_rbf_seed_17_pooled_out_of_fold_predictions.csv")
    print(" - svm_rbf_fold_0_seed_17_test_predictions.csv")
    print()
    print(
        "Set SVM_SOURCE_OVERRIDE in the configuration cell to the exact "
        "prediction file or containing directory, then rerun."
    )
    raise FileNotFoundError("RBF-SVM prediction outputs were not found.")

print("\nUsing SVM source:")
print(svm_source)

svm_raw = load_svm_predictions(svm_source)

svm_participant = aggregate_seed_ensemble(
    svm_raw,
    probability_column="PROB_PMCI",
    seeds=COMMON_SEEDS,
)

svm_participant = svm_participant.rename(
    columns={
        "P_MEAN": "SVM_P_MEAN",
        "P_SD": "SVM_P_SD",
        "PRED_CLASS": "SVM_PRED_CLASS",
        "CORRECT": "SVM_CORRECT",
        "CORRECT_CLASS_PROBABILITY": "SVM_CORRECT_CLASS_PROBABILITY",
        "N_SEEDS": "N_SVM_SEEDS",
    }
)

print("\nRBF-SVM three-seed ensemble:")
print(
    "Pooled ROC AUC:",
    f"{safe_auc(svm_participant['TARGET'], svm_participant['SVM_P_MEAN']):.6f}",
)
print(
    "Pooled AP:",
    f"{safe_ap(svm_participant['TARGET'], svm_participant['SVM_P_MEAN']):.6f}",
)

display(svm_participant.head())


## 3. Load neural predictions and construct matched three-seed ensembles

The **primary disagreement comparison** uses exactly the same stochastic seeds as the SVM: **17, 42, 73**.

For context, the notebook also calculates the five-seed neural ensemble from **17, 42, 73, 101, 137**.


In [ ]:
neural_matched = {}
neural_five_seed = {}
neural_raw_cache = {}

for architecture_name, root in ARCHITECTURE_ROOTS.items():
    try:
        raw_all = load_neural_predictions(
            root,
            architecture_name,
            seeds=ALL_NEURAL_SEEDS,
        )
        neural_raw_cache[architecture_name] = raw_all

        matched = aggregate_seed_ensemble(
            raw_all,
            probability_column="FINAL_P_pMCI",
            seeds=COMMON_SEEDS,
            uncertainty_column="FINAL_UNCERTAINTY",
        )

        five = aggregate_seed_ensemble(
            raw_all,
            probability_column="FINAL_P_pMCI",
            seeds=ALL_NEURAL_SEEDS,
            uncertainty_column="FINAL_UNCERTAINTY",
        )

        neural_matched[architecture_name] = matched
        neural_five_seed[architecture_name] = five

        print(
            f"{architecture_name}: "
            f"matched-3 AUC={safe_auc(matched['TARGET'], matched['P_MEAN']):.6f}; "
            f"five-seed AUC={safe_auc(five['TARGET'], five['P_MEAN']):.6f}"
        )

    except FileNotFoundError as exc:
        print(f"Skipping {architecture_name}: {exc}")

if not neural_matched:
    raise RuntimeError("No neural architecture predictions were available.")


## 4. Head-to-head SVM vs neural disagreement

For each architecture this reports:

- `both_correct`
- `svm_rescues_neural`
- `neural_rescues_svm`
- `both_wrong`

The two most interpretable rates are:

- **SVM rescue rate among neural errors**  
  = SVM-right / neural-wrong participants
- **Neural rescue rate among SVM errors**  
  = neural-right / SVM-wrong participants

A large SVM rescue rate is direct evidence of error complementarity.


In [ ]:
head_to_head_tables = {}
summary_rows = []

for architecture_name, neural in neural_matched.items():
    merged = neural.merge(
        svm_participant[
            [
                "RID",
                "TARGET",
                "FOLD",
                "SVM_P_MEAN",
                "SVM_P_SD",
                "SVM_PRED_CLASS",
                "SVM_CORRECT",
                "SVM_CORRECT_CLASS_PROBABILITY",
            ]
        ],
        on="RID",
        how="inner",
        suffixes=("_NEURAL", "_SVM"),
        validate="one_to_one",
    )

    if len(merged) != 544:
        warnings.warn(
            f"{architecture_name}: matched {len(merged)} participants, expected 544."
        )

    if not (
        merged["TARGET_NEURAL"].astype(int)
        == merged["TARGET_SVM"].astype(int)
    ).all():
        raise ValueError(
            f"{architecture_name}: neural and SVM targets do not match."
        )

    if not (
        merged["FOLD_NEURAL"].astype(int)
        == merged["FOLD_SVM"].astype(int)
    ).all():
        raise ValueError(
            f"{architecture_name}: neural and SVM fold labels do not match."
        )

    merged = merged.rename(
        columns={
            "TARGET_NEURAL": "TARGET",
            "FOLD_NEURAL": "FOLD",
            "P_MEAN": "NEURAL_P_MEAN",
            "P_SD": "NEURAL_P_SD",
            "PRED_CLASS": "NEURAL_PRED_CLASS",
            "CORRECT": "NEURAL_CORRECT",
            "CORRECT_CLASS_PROBABILITY":
                "NEURAL_CORRECT_CLASS_PROBABILITY",
        }
    )

    merged = merged.drop(
        columns=["TARGET_SVM", "FOLD_SVM"],
        errors="ignore",
    )

    merged["DISAGREEMENT_GROUP"] = np.select(
        [
            (merged["NEURAL_CORRECT"] == 1)
            & (merged["SVM_CORRECT"] == 1),

            (merged["NEURAL_CORRECT"] == 0)
            & (merged["SVM_CORRECT"] == 1),

            (merged["NEURAL_CORRECT"] == 1)
            & (merged["SVM_CORRECT"] == 0),
        ],
        [
            "both_correct",
            "svm_rescues_neural",
            "neural_rescues_svm",
        ],
        default="both_wrong",
    )

    counts = (
        merged["DISAGREEMENT_GROUP"]
        .value_counts()
        .reindex(
            [
                "both_correct",
                "svm_rescues_neural",
                "neural_rescues_svm",
                "both_wrong",
            ],
            fill_value=0,
        )
    )

    neural_errors = int((merged["NEURAL_CORRECT"] == 0).sum())
    svm_errors = int((merged["SVM_CORRECT"] == 0).sum())

    svm_rescues = int(counts["svm_rescues_neural"])
    neural_rescues = int(counts["neural_rescues_svm"])

    summary_rows.append({
        "ARCHITECTURE": architecture_name,
        "N": len(merged),

        "NEURAL_MATCHED_3SEED_AUC":
            safe_auc(merged["TARGET"], merged["NEURAL_P_MEAN"]),

        "SVM_3SEED_AUC":
            safe_auc(merged["TARGET"], merged["SVM_P_MEAN"]),

        "BOTH_CORRECT":
            int(counts["both_correct"]),

        "SVM_RESCUES_NEURAL":
            svm_rescues,

        "NEURAL_RESCUES_SVM":
            neural_rescues,

        "BOTH_WRONG":
            int(counts["both_wrong"]),

        "NEURAL_ERRORS":
            neural_errors,

        "SVM_ERRORS":
            svm_errors,

        "SVM_RESCUE_RATE_AMONG_NEURAL_ERRORS":
            svm_rescues / neural_errors if neural_errors else np.nan,

        "NEURAL_RESCUE_RATE_AMONG_SVM_ERRORS":
            neural_rescues / svm_errors if svm_errors else np.nan,

        "PROBABILITY_CORRELATION":
            merged["NEURAL_P_MEAN"].corr(merged["SVM_P_MEAN"]),
    })

    merged["ARCHITECTURE"] = architecture_name
    head_to_head_tables[architecture_name] = merged

comparison_summary = pd.DataFrame(summary_rows)

display(comparison_summary)

comparison_summary.to_csv(
    TABLES_DIR / "svm_vs_neural_summary.csv",
    index=False,
)


## 5. Breakdown by fold and true class

This checks whether the SVM's rescues are concentrated in particular outer folds or in sMCI / pMCI.


In [ ]:
fold_class_rows = []

for architecture_name, merged in head_to_head_tables.items():
    for fold in FOLDS:
        for target in [0, 1]:
            g = merged.loc[
                (merged["FOLD"] == fold)
                & (merged["TARGET"] == target)
            ].copy()

            if len(g) == 0:
                continue

            counts = g["DISAGREEMENT_GROUP"].value_counts()

            neural_errors = int((g["NEURAL_CORRECT"] == 0).sum())
            svm_errors = int((g["SVM_CORRECT"] == 0).sum())

            svm_rescue = int(counts.get("svm_rescues_neural", 0))
            neural_rescue = int(counts.get("neural_rescues_svm", 0))

            fold_class_rows.append({
                "ARCHITECTURE": architecture_name,
                "FOLD": fold,
                "TARGET": target,
                "TARGET_LABEL": "sMCI" if target == 0 else "pMCI",
                "N": len(g),
                "NEURAL_AUC_WITHIN_GROUP":
                    safe_auc(g["TARGET"], g["NEURAL_P_MEAN"]),
                "SVM_AUC_WITHIN_GROUP":
                    safe_auc(g["TARGET"], g["SVM_P_MEAN"]),
                "NEURAL_ERRORS": neural_errors,
                "SVM_ERRORS": svm_errors,
                "SVM_RESCUES_NEURAL": svm_rescue,
                "NEURAL_RESCUES_SVM": neural_rescue,
                "BOTH_WRONG": int(
                    counts.get("both_wrong", 0)
                ),
                "SVM_RESCUE_RATE_AMONG_NEURAL_ERRORS":
                    svm_rescue / neural_errors
                    if neural_errors else np.nan,
            })

fold_class_summary = pd.DataFrame(fold_class_rows)

display(
    fold_class_summary[
        [
            "ARCHITECTURE",
            "FOLD",
            "TARGET_LABEL",
            "N",
            "NEURAL_ERRORS",
            "SVM_ERRORS",
            "SVM_RESCUES_NEURAL",
            "NEURAL_RESCUES_SVM",
            "BOTH_WRONG",
            "SVM_RESCUE_RATE_AMONG_NEURAL_ERRORS",
        ]
    ]
)

fold_class_summary.to_csv(
    TABLES_DIR / "svm_vs_neural_by_fold_and_target.csv",
    index=False,
)


## 6. Participants all three neural architectures miss

This is the strongest version of the complementarity question:

> Among participants that **small, medium, and full** neural ensembles all get wrong, how many does the SVM get right?

The comparison remains on the **three common seeds** for fairness.


In [ ]:
if len(head_to_head_tables) >= 3:
    neural_error_sets = {}

    for architecture_name, merged in head_to_head_tables.items():
        neural_error_sets[architecture_name] = set(
            merged.loc[
                merged["NEURAL_CORRECT"] == 0,
                "RID",
            ].tolist()
        )

    common_neural_wrong = set.intersection(
        *neural_error_sets.values()
    )

    reference_name = (
        "medium_2.51M"
        if "medium_2.51M" in head_to_head_tables
        else list(head_to_head_tables)[0]
    )

    reference = head_to_head_tables[reference_name]

    common_wrong_table = reference.loc[
        reference["RID"].isin(common_neural_wrong)
    ].copy()

    n_common_wrong = len(common_wrong_table)
    n_svm_rescued = int(
        common_wrong_table["SVM_CORRECT"].sum()
    )
    n_both_wrong = int(
        (common_wrong_table["SVM_CORRECT"] == 0).sum()
    )

    print("Participants wrong for ALL available neural architectures:")
    print(n_common_wrong)

    print("\nOf those, SVM correct:")
    print(n_svm_rescued)

    print("\nOf those, SVM also wrong:")
    print(n_both_wrong)

    if n_common_wrong:
        print("\nSVM rescue fraction among cross-architecture neural failures:")
        print(f"{n_svm_rescued / n_common_wrong:.3%}")

    common_wrong_table.to_csv(
        TABLES_DIR
        / "participants_wrong_all_neural_architectures_with_svm_status.csv",
        index=False,
    )

    display(
        common_wrong_table[
            [
                "RID",
                "FOLD",
                "TARGET",
                "NEURAL_P_MEAN",
                "SVM_P_MEAN",
                "NEURAL_CORRECT",
                "SVM_CORRECT",
                "DISAGREEMENT_GROUP",
            ]
        ]
        .sort_values(
            ["SVM_CORRECT", "FOLD"],
            ascending=[False, True],
        )
        .reset_index(drop=True)
        .head(50)
    )

else:
    print(
        "Fewer than three neural architecture result sets were available; "
        "skipping the all-three-neural-failures analysis."
    )


## 7. Join participant characteristics and neural uncertainty

This section asks **what distinguishes**:

- `svm_rescues_neural`
- `both_wrong`
- `neural_rescues_svm`
- `both_correct`

It uses the medium architecture as the primary neural model when available, then joins the prepared held-out participant covariates.

This is descriptive only — not a feature-selection stage.


In [ ]:
def load_oof_test_covariates():
    frames = []

    for fold in FOLDS:
        path = (
            INPUT_ROOT
            / f"mci_prognosis_outer_fold_{fold}_final_task_ready.csv"
        )

        if not path.exists():
            raise FileNotFoundError(path)

        frame = pd.read_csv(path)

        require_columns(
            frame,
            ["RID", "DATA_ROLE", "MODEL_TARGET"],
            str(path),
        )

        test = frame.loc[
            frame["DATA_ROLE"].astype(str).str.lower() == "test"
        ].copy()

        test["FOLD_FROM_FILE"] = fold
        frames.append(test)

    oof = pd.concat(frames, ignore_index=True)

    if oof["RID"].duplicated().any():
        raise ValueError(
            "Prepared test folds are not disjoint by RID."
        )

    return oof

def standardized_mean_difference(x_a, x_b):
    a = pd.to_numeric(x_a, errors="coerce").dropna().to_numpy(float)
    b = pd.to_numeric(x_b, errors="coerce").dropna().to_numpy(float)

    if len(a) < 2 or len(b) < 2:
        return np.nan

    pooled_sd = np.sqrt(
        (np.var(a, ddof=1) + np.var(b, ddof=1)) / 2.0
    )

    if pooled_sd == 0 or not np.isfinite(pooled_sd):
        return np.nan

    return float(
        (np.mean(a) - np.mean(b)) / pooled_sd
    )

primary_architecture = (
    "medium_2.51M"
    if "medium_2.51M" in head_to_head_tables
    else list(head_to_head_tables)[0]
)

primary = head_to_head_tables[primary_architecture].copy()

# Add neural auxiliary quantities from the matched-seed raw predictions.
raw_primary = neural_raw_cache[primary_architecture]
aux = aggregate_seed_ensemble(
    raw_primary,
    probability_column="FINAL_P_pMCI",
    seeds=COMMON_SEEDS,
    uncertainty_column="FINAL_UNCERTAINTY",
)

# Rename auxiliary columns to avoid collision.
aux_rename = {}
for col in aux.columns:
    if col not in ["RID", "TARGET", "FOLD"]:
        aux_rename[col] = f"AUX__{col}"

aux = aux.rename(columns=aux_rename)

primary = primary.merge(
    aux.drop(columns=["TARGET", "FOLD"]),
    on="RID",
    how="left",
    validate="one_to_one",
)

oof_covariates = load_oof_test_covariates()

branch_mask_cols = [
    c for c in oof_covariates.columns
    if c.startswith("BRANCH_MASK__")
]

feature_mask_cols = [
    c for c in oof_covariates.columns
    if c.startswith("FEATURE_MASK__")
]

scaled_continuous_cols = [
    c for c in oof_covariates.columns
    if c.endswith("__Z")
]

audit_cols = [
    c for c in [
        "PTID",
        "CLINICAL_GROUP",
        "MCI_TRAJECTORY_LABEL",
    ]
    if c in oof_covariates.columns
]

keep_cols = (
    ["RID"]
    + audit_cols
    + branch_mask_cols
    + feature_mask_cols
    + scaled_continuous_cols
)

keep_cols = list(dict.fromkeys(keep_cols))

primary = primary.merge(
    oof_covariates[keep_cols],
    on="RID",
    how="left",
    validate="one_to_one",
)

if feature_mask_cols:
    primary["OBSERVED_FEATURE_COUNT"] = (
        primary[feature_mask_cols].sum(axis=1)
    )
    primary["MISSING_FEATURE_COUNT"] = (
        len(feature_mask_cols)
        - primary["OBSERVED_FEATURE_COUNT"]
    )
    primary["FEATURE_AVAILABILITY_FRACTION"] = (
        primary["OBSERVED_FEATURE_COUNT"]
        / len(feature_mask_cols)
    )

if branch_mask_cols:
    primary["AVAILABLE_BRANCH_COUNT"] = (
        primary[branch_mask_cols].sum(axis=1)
    )

print("Primary architecture:", primary_architecture)

diagnostic_means = [
    c for c in [
        "NEURAL_P_SD",
        "AUX__UNCERTAINTY_MEAN",
        "AUX__TMC_UNCERTAINTY__mean",
        "AUX__THREE_MT_UNCERTAINTY__mean",
        "AUX__TMC_P_pMCI__mean",
        "AUX__THREE_MT_P_pMCI__mean",
        "MISSING_FEATURE_COUNT",
        "FEATURE_AVAILABILITY_FRACTION",
        "AVAILABLE_BRANCH_COUNT",
    ]
    if c in primary.columns
]

if diagnostic_means:
    disagreement_profile = (
        primary
        .groupby("DISAGREEMENT_GROUP", observed=True)[diagnostic_means]
        .mean()
        .T
    )

    print("\nMean diagnostic quantities by disagreement group:")
    display(disagreement_profile)

    disagreement_profile.to_csv(
        TABLES_DIR / "disagreement_group_diagnostic_profile.csv"
    )

# Continuous-variable ranking:
# SVM-rescued neural errors vs participants both models get wrong.
svm_rescue_group = primary.loc[
    primary["DISAGREEMENT_GROUP"] == "svm_rescues_neural"
]

both_wrong_group = primary.loc[
    primary["DISAGREEMENT_GROUP"] == "both_wrong"
]

difference_rows = []

for col in scaled_continuous_cols:
    smd = standardized_mean_difference(
        svm_rescue_group[col],
        both_wrong_group[col],
    )

    difference_rows.append({
        "VARIABLE": col,
        "MEAN_SVM_RESCUES_NEURAL":
            pd.to_numeric(
                svm_rescue_group[col],
                errors="coerce",
            ).mean(),
        "MEAN_BOTH_WRONG":
            pd.to_numeric(
                both_wrong_group[col],
                errors="coerce",
            ).mean(),
        "SMD_RESCUED_MINUS_BOTH_WRONG": smd,
        "ABS_SMD": abs(smd) if np.isfinite(smd) else np.nan,
        "N_SVM_RESCUES_NEURAL":
            int(pd.to_numeric(
                svm_rescue_group[col],
                errors="coerce",
            ).notna().sum()),
        "N_BOTH_WRONG":
            int(pd.to_numeric(
                both_wrong_group[col],
                errors="coerce",
            ).notna().sum()),
    })

rescued_vs_both_wrong_features = (
    pd.DataFrame(difference_rows)
    .sort_values("ABS_SMD", ascending=False)
    .reset_index(drop=True)
)

print("\nVariables differing most between SVM-rescued neural failures and both-wrong cases:")
display(rescued_vs_both_wrong_features.head(20))

rescued_vs_both_wrong_features.to_csv(
    TABLES_DIR / "svm_rescued_vs_both_wrong_continuous_smd.csv",
    index=False,
)

primary.to_csv(
    TABLES_DIR / "primary_participant_level_svm_vs_neural.csv",
    index=False,
)


## 8. Probability-space visualization

This plot is useful for seeing whether the SVM and neural model mostly agree on ranking, or whether the rescued cases occupy a distinct disagreement region.

The diagonal represents equal pMCI probability from both models.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

groups = [
    "both_correct",
    "svm_rescues_neural",
    "neural_rescues_svm",
    "both_wrong",
]

for group in groups:
    g = primary.loc[
        primary["DISAGREEMENT_GROUP"] == group
    ]

    ax.scatter(
        g["NEURAL_P_MEAN"],
        g["SVM_P_MEAN"],
        label=f"{group} (n={len(g)})",
        alpha=0.70,
    )

ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1)

ax.axvline(0.5, linestyle=":", linewidth=1)
ax.axhline(0.5, linestyle=":", linewidth=1)

ax.set_xlabel(
    f"{primary_architecture} matched-3-seed mean pMCI probability"
)
ax.set_ylabel(
    "RBF-SVM matched-3-seed mean pMCI probability"
)
ax.set_title("RBF-SVM vs neural participant predictions")
ax.legend()
fig.tight_layout()

scatter_path = FIGURES_DIR / "svm_vs_neural_probability_scatter.png"
fig.savefig(scatter_path, dpi=180, bbox_inches="tight")
plt.show()

print("Saved:", scatter_path)


## 9. Final decision-oriented summary

This does **not** decide a fusion weight. It prints the quantities that tell us whether fusion is worth testing next.

A useful rule of thumb for interpretation:

- **many `svm_rescues_neural` + relatively few `both_wrong`** → strong complementarity; fusion is promising;
- **mostly `both_wrong`** → both models fail on the same cases; investigate participant phenotype / trajectory / labels rather than expecting fusion alone to solve it;
- substantial rescue in **both directions** → especially strong case for a carefully validated fusion/meta-model.


In [ ]:
print("=" * 90)
print("RBF-SVM VS NEURAL PARTICIPANT DISAGREEMENT — FINAL SUMMARY")
print("=" * 90)

display(comparison_summary)

print("\nPrimary architecture:", primary_architecture)

primary_counts = (
    primary["DISAGREEMENT_GROUP"]
    .value_counts()
    .reindex(
        [
            "both_correct",
            "svm_rescues_neural",
            "neural_rescues_svm",
            "both_wrong",
        ],
        fill_value=0,
    )
)

print("\nPrimary disagreement counts:")
print(primary_counts.to_string())

neural_errors = int((primary["NEURAL_CORRECT"] == 0).sum())
svm_errors = int((primary["SVM_CORRECT"] == 0).sum())

svm_rescues = int(primary_counts["svm_rescues_neural"])
neural_rescues = int(primary_counts["neural_rescues_svm"])

print("\nNeural errors:", neural_errors)
print("SVM errors:", svm_errors)

if neural_errors:
    print(
        "SVM rescue rate among neural errors:",
        f"{svm_rescues / neural_errors:.3%}",
    )

if svm_errors:
    print(
        "Neural rescue rate among SVM errors:",
        f"{neural_rescues / svm_errors:.3%}",
    )

print(
    "\nProbability correlation:",
    f"{primary['NEURAL_P_MEAN'].corr(primary['SVM_P_MEAN']):.4f}",
)

if len(head_to_head_tables) >= 3:
    neural_error_sets = {
        name: set(
            table.loc[
                table["NEURAL_CORRECT"] == 0,
                "RID",
            ].tolist()
        )
        for name, table in head_to_head_tables.items()
    }

    common_wrong = set.intersection(
        *neural_error_sets.values()
    )

    common_table = primary.loc[
        primary["RID"].isin(common_wrong)
    ]

    print("\nParticipants missed by every neural architecture:")
    print(len(common_table))

    print("...rescued by SVM:")
    print(int(common_table["SVM_CORRECT"].sum()))

    print("...also missed by SVM:")
    print(int((common_table["SVM_CORRECT"] == 0).sum()))

print("\nSaved tables:")
for path in sorted(TABLES_DIR.glob("*.csv")):
    print(" -", path)

print("\nSaved figures:")
for path in sorted(FIGURES_DIR.glob("*.png")):
    print(" -", path)

print(
    "\nNext methodological step if complementarity is strong: "
    "construct a separate fusion experiment where the SVM–TMC/Hybrid "
    "weight is selected strictly from training/validation predictions, "
    "then evaluated once on the held-out test fold."
)
